# 11 — kvpress: LongBench Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on two
[LongBench](https://github.com/THUDM/LongBench) tasks using
[kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B:

- **gov_report** — government report summarization (~8.7K words). Generates
  one-page summaries, providing **high decoding stress**.
- **hotpotqa** — multi-document QA (~9.2K words). Tests multi-hop reasoning
  across scattered paragraphs.

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

Scoring uses the HuggingFace `evaluate` library:
- gov_report: ROUGE-L (via `evaluate.load("rouge")`)
- hotpotqa: F1 (via `evaluate.load("squad")`)

Results are saved to `results/kvpress_longbench/` for comparison in later notebooks.

## Configuration

In [ ]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

LONGBENCH_TASKS = ["gov_report", "hotpotqa"]

MAX_NEW_TOKENS = {
    "gov_report": 512,
    "hotpotqa": 64,
}

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

## 1. Load Model

In [ ]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2. Load LongBench Datasets

In [ ]:
from datasets import load_dataset

longbench_datasets = {}
for task_name in LONGBENCH_TASKS:
    ds = load_dataset("THUDM/LongBench", task_name, split="test")
    longbench_datasets[task_name] = ds
    print(f"{task_name}: {len(ds)} examples")
    print(f"  Sample input: {ds[0]['input'][:120]}...")
    print(f"  Context length (words): {ds[0]['length']}")
    print(f"  Answers: {ds[0]['answers'][:2]}")
    print()

## 3. Load Scoring Metrics

In [ ]:
import evaluate

rouge_metric = evaluate.load("rouge")
squad_metric = evaluate.load("squad")

TASK_METRICS = {
    "gov_report": rouge_metric,
    "hotpotqa": squad_metric,
}

print("Loaded scoring metrics:")
print("  gov_report → ROUGE-L")
print("  hotpotqa   → SQuAD F1")

## 4. Run Inference

For each (algorithm, compression_ratio, LongBench task) combination, run all
examples through the kvpress pipeline. Predictions are collected for scoring
in the next section.

In [ ]:
import time

all_results = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    for task_name in LONGBENCH_TASKS:
        ds = longbench_datasets[task_name]
        max_tokens = MAX_NEW_TOKENS[task_name]
        label = f"{press_name} | ratio={ratio} | task={task_name}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(ds)} examples)")
        print(f"{'='*60}")

        torch.cuda.reset_peak_memory_stats()
        start_config = time.perf_counter()

        for i, row in enumerate(ds):
            start = time.perf_counter()

            kwargs = dict(
                question=row["input"],
                max_new_tokens=max_tokens,
            )
            if press is not None:
                kwargs["press"] = press

            output = pipe(row["context"], **kwargs)
            elapsed = time.perf_counter() - start

            all_results.append({
                "framework": "kvpress",
                "press": press_name,
                "compression_ratio": ratio,
                "longbench_task": task_name,
                "predicted_answer": output["answer"],
                "reference_answers": row["answers"],
                "elapsed_sec": round(elapsed, 3),
            })

            if (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(ds)}")

        config_elapsed = time.perf_counter() - start_config
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Done: {config_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

        torch.cuda.empty_cache()

print(f"\nTotal results: {len(all_results)}")

## 5. Score & Results

Score predictions using HuggingFace `evaluate`:
- **gov_report**: ROUGE-L F-measure
- **hotpotqa**: SQuAD token-level F1

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []

for (press, ratio, task_name), group in df.groupby(
    ["press", "compression_ratio", "longbench_task"]
):
    preds = group["predicted_answer"].tolist()
    refs = group["reference_answers"].tolist()

    if task_name == "gov_report":
        flat_refs = [r[0] if isinstance(r, list) else r for r in refs]
        result = rouge_metric.compute(predictions=preds, references=flat_refs)
        score = result["rougeL"]
        metric_name = "rougeL"
    else:
        squad_preds = [
            {"prediction_text": p, "id": str(i)}
            for i, p in enumerate(preds)
        ]
        squad_refs = [
            {"answers": {"text": r if isinstance(r, list) else [r],
                         "answer_start": [0] * (len(r) if isinstance(r, list) else 1)},
             "id": str(i)}
            for i, r in enumerate(refs)
        ]
        result = squad_metric.compute(predictions=squad_preds, references=squad_refs)
        score = result["f1"]
        metric_name = "f1"

    key = f"{press}__{ratio}__{task_name}"
    all_metrics[key] = {metric_name: round(score, 4)}
    mean_time = group["elapsed_sec"].mean()
    rows.append({
        "press": press, "compression_ratio": ratio,
        "longbench_task": task_name, "score": round(score, 4),
        "metric": metric_name, "mean_time": round(mean_time, 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 6. Save Results

In [ ]:
import json

os.makedirs("results/kvpress_longbench", exist_ok=True)

predictions_path = "results/kvpress_longbench/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_longbench/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")